## Preparation

In [0]:
%sql
-- initializing the required schemas / layers 
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
import pandas as pd
from pyspark.sql.functions import col, expr, from_json, first, year, count, row_number, desc
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window

## Bronze Layer

### Processing **itme.csv**

In [0]:
# reading the item.csv file from S3 bucket
item_bronze_source = pd.read_csv("https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/item.csv", dtype=str)
item_bronze = spark.createDataFrame(item_bronze_source)

# saving item_bronze (item.csv) into the bronze layer
item_bronze.write.mode("overwrite").saveAsTable("bronze.item")

### Processing **event.csv**

In [0]:
# reading the event.csv file from S3 bucket
event_bronze_source = pd.read_csv("https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/event.csv", dtype=str)
event_bronze = spark.createDataFrame(event_bronze_source)

# saving event_bronze (event.csv) into the bronze layer
event_bronze.write.mode("overwrite").saveAsTable("bronze.event")

## Silver Layer

### Processing **itme.csv**

In [0]:
item_silver = spark.read.table("workspace.bronze.item")

# changing data types (created_at, id, price)
item_silver = item_silver.withColumn("created_at", col("created_at").cast("timestamp")) # created_at
item_silver = item_silver.withColumn("id", col("id").cast("double").cast("int")) # id
item_silver = item_silver.withColumn("price", expr("round(price, 2)")) # price

# saving the silver layer table
item_silver.write \
    .partitionBy("category") \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.item")

### Processing **event.csv**

In [0]:
event_silver = spark.read.table("workspace.bronze.event")

# keeping the naming convention (event.payload -> event_payload)
event_silver = event_silver.selectExpr(
    "event_id",
    "event_time",
    "user_id",
    "`event.payload` as event_payload"
)

# setting up schema for flattening
schema = StructType([
    StructField("event_name", StringType(), True),
    StructField("platform", StringType(), True),
    StructField("parameter_name", StringType(), True),
    StructField("parameter_value", StringType(), True)
])

# creating a proper nested column (string -> json)
event_silver = event_silver.withColumn(
    "event_payload",
    from_json(col("event_payload"), schema)
)

# flattening the whole df
event_silver = event_silver.select(
    "event_id",
    "event_time",
    "user_id",
    "event_payload.*"
)

# changing data types (event_time, user_id)
event_silver = event_silver.withColumn("event_time", col("event_time").cast("timestamp")) # event_time
event_silver = event_silver.withColumn("user_id", col("user_id").cast("double").cast("int")) # user_id

### events fact table

In [0]:
events = event_silver.select(
    "event_id",
    "event_time",
    "user_id",
    "event_name",
    "platform"
).dropDuplicates(["event_id"])

events.write.mode("overwrite").saveAsTable("workspace.silver.events")

### event_items dim table

In [0]:
event_items = event_silver.filter(col("parameter_name") == "item_id") \
    .select(
        "event_id",
        col("parameter_value").cast("int").alias("item_id")
    )
event_items.write.mode("overwrite").saveAsTable("workspace.silver.event_items")

### event_referrers dim table

In [0]:
event_referrers = event_silver.filter(col("parameter_name") == "referrer") \
    .select(
        "event_id",
        col("parameter_value").alias("referrer")
    )

event_referrers.write.mode("overwrite").saveAsTable("workspace.silver.event_referrers")

### event_tests dim table

In [0]:
event_tests = event_silver.filter(
    col("parameter_name").isin("test_assignment", "test_id")
).groupBy("event_id").pivot("parameter_name").agg(first("parameter_value"))

event_tests = event_tests.withColumn("test_id", col("test_id").cast("int"))

event_tests.write.mode("overwrite").saveAsTable("workspace.silver.event_tests")

### event_viewed_users dim table


In [0]:
event_viewed_users = event_silver.filter(col("parameter_name") == "viewed_user_id") \
    .select(
        "event_id",
        col("parameter_value").cast("int").alias("viewed_user_id")
    )

event_viewed_users.write.mode("overwrite").saveAsTable("workspace.silver.event_viewed_users")

## Gold Layer

In [0]:
# taking events and event_itmes tables
events = spark.read.table("workspace.silver.events")
event_items = spark.read.table("workspace.silver.event_items")

# joining the loaded dfs
item_views = events.join(
    event_items,
    on="event_id",
    how="inner"
).filter(
    col("event_name") == "view_item" # filtering only view_item events
)

# creating new column for year in the joint df
item_views = item_views.withColumn("year", year(col("event_time")))

# counting the views per item per year
item_year_views = item_views.groupBy(
    "item_id",
    "year"
).agg(
    count("*").alias("total_item_views")
)

# ranking items in each year
rank_window = Window.partitionBy("year").orderBy(desc("total_item_views"))

item_year_views = item_year_views.withColumn(
    "item_rank",
    row_number().over(rank_window)
)

# finding most used platforms
platform_counts = item_views.groupBy(
    "item_id",
    "year",
    "platform"
).agg(
    count("*").alias("platform_views")
)

platform_window = Window.partitionBy(
    "item_id", "year"
).orderBy(desc("platform_views"))

top_platform = platform_counts.withColumn(
    "platform_rank",
    row_number().over(platform_window)
).filter(
    col("platform_rank") == 1
).select(
    "item_id",
    "year",
    col("platform").alias("most_used_platform")
)

# creating the final df
top_item = item_year_views.join(
    top_platform,
    on=["item_id", "year"],
    how="left"
).select(
    "item_id",
    "year",
    "total_item_views",
    "item_rank",
    "most_used_platform"
)

top_item.write.mode("overwrite").saveAsTable("workspace.gold.top_item")

In [0]:
top_item.display()